# 🎯 [MNT04_M09] AAR 기반 예지정비 오경보 분석 및 이상 감지 임계 파라미터 재보정
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT04/mnt04_m09_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT04_M09 |
| **미션 주제** | AAR 기반 예지정비 오경보 분석 및 이상 감지 임계 파라미터 재보정 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
항공·기갑 장비의 엔진과 구동계는 고장이 나기 전에 진동, 온도, 압력에 미세한 변화가 먼저 나타나지만, 고정 임계값만으로는 고부하 운용일을 이상으로 오인하거나 초기 열화를 놓쳐 수리부속을 제때 확보하지 못한다. 본 프로젝트는 고장 실적과 수리부속 소모를 연결해 정비와 부품 관리의 연계 문제를 학습하는 것을 목표로, 교육용 가상 기동체 48대의 센서 시계열과 부품 소모를 코드로 만들어 분석한다. 센서 데이터셋 정의, 잡음 제거와 FFT 특징량 추출, 임계값 Baseline과 부품 소모 파레토 분석, 오토인코더 기반 미세 변화 감지와 파생 피처 모델, 잔존수명(RUL) 예측과 이상 신뢰도 점수, 설명가능한 경보 화면, 검사관 승인(HITL) 기반 조기 입고 시뮬레이터, 대용량 스트림 스트레스 시험, AAR 오경보 분석과 임계 재보정, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 고장 전에 이상 징후를 일찍 잡아 정비 시점과 수리부속 소요를 예고하되, 조기 입고 결정은 검사관이 근거를 보고 내리도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 정비 야전(정비대대, 군수지원사령부)에 즉시 도입 가능한 "완성형 국방 예지정비(PdM) 및 PBL 품목관리 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 도메인 및 데이터 규격서** | 군수 도메인 분석서 및 데이터 딕셔너리(`docs/mnt04_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mnt04_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **센서 데이터 정제 및 전처리 모듈** | 고주파 센서 신호 잡음 제거, 결측치 보정 및 FFT 특징 추출 파이프라인(`src/etl/mnt04_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **탐색적 데이터 분석(EDA) & Baseline** | 3-시그마 임계값 Baseline 및 부품 소모 파레토 분석(`notebooks/mnt04_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 오토인코더 기반 미세 진동 변화 감지 및 파생 피처 엔지니어링 모듈(`src/features/mnt04_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 잔존수명(RUL) 엔진** | 부품 RUL 예측 앙상블 모델 및 종합 상태 이상 신뢰도 엔진(`src/models/mnt04_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | 오토인코더 재구성 오차 분해 및 XAI 원인 설명기(`src/explainability/mnt04_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **HITL 조기 입고 시뮬레이터** | 검사관 HITL 승인 큐, 감사 추적 및 조기 정비 시뮬레이터(`src/optimization/mnt04_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **스트림 주입 & 관제 대시보드** | 대용량 센서 스트림 주입 스트레스 시험 및 실시간 관제 대시보드(`src/dashboard/mnt04_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 오경보 원인 분석 및 임계 파라미터 재보정 ROI 보고서(`reports/mnt04_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 데이터 기반 챔피언-도전자 자동 재학습 및 E2E 통합 파이프라인(`pipeline/mnt04_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 예지정비 오경보 분석 및 이상 감지 임계 파라미터 재보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 예지정비 오경보를 분석하고 임계 파라미터를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR 기반으로 예지정비 오경보를 원인별로 분석하고 이상 감지 임계 파라미터를 재보정해 효과를 지표로 산출한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AAR 기간 오경보를 고부하 운용일, 외기 급변, 센서 스파이크, 열화 초기 경계 등으로 분류하고, 정상 분위수·연속 일수·고부하일 임계 완화를 AAR 기간 자료만으로 격자 탐색해 재보정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `reports/mnt04_roi_analysis.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추정 산식의 객관성 및 군 재정/군수 기획 부서 설득력

#### 미션 PASS 조건

- 정량 기준: 비용 절감액, 시간 단축률 등 3대 정량 효과 지표 산출


- 진행 기준: ROI 분석 보고서 완성 및 주요 지표 민감도 분석 수록


- 검증 방법: 회계/기획 기준 모델과의 비교 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# ============================================================================
# [미션 09 산출물 파일 연계]: reports/mnt04_roi_analysis.md
# (선택) 로컬 모듈 파일로 추출 저장 시: %%writefile reports/mnt04_roi_analysis.md
# ============================================================================
"""AAR 기간의 예지정비 오경보를 원인(고부하 운용일·외기 급변·센서 스파이크·기타)별로 분석하고, 정상 분위수·연속 일수·고부하일 임계 완화 파라미터를 AAR 기간 자료로만 격자 탐색해 재보정한 뒤 검증 기간에서 오경보·고장 전 확보 시간 변화와 비용 절감·점검 시간 단축·확보 시간 증가의 3대 정량 효과 지표를 가정 단가로 산출한다."""
import itertools
import numpy as np
import pandas as pd
from scipy.stats import kurtosis
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 기동체 48대의 엔진·구동계 진동·온도·압력 센서 자료 생성기 (실제 장비·부대·정비 실적 없음)
# 개체마다 정상 -> 열화 -> 고장(수명종료)까지 하루 1회 측정(진동 파형 256표본 + 온도 + 압력 + 부하)을 만든다.
# 열화 유형 4가지: bearing(베어링), gear(기어), oil_pump(오일 펌프), cooling(냉각 팬 모터)
# ---------------------------------------------------------------------------
FS, NS = 1024, 256                              # 가상 표본화 주파수(Hz), 파형 1개당 표본 수
TT = np.arange(NS) / FS
MODES = {  # 열화 유형: (발생 확률, 평균 수명(일), 교체 수리부속, 가상 단가지수)
    "bearing": (0.40, 160, "주 베어링 세트", 6.0), "gear": (0.25, 175, "구동 기어 세트", 9.0),
    "oil_pump": (0.20, 150, "오일 펌프", 4.0), "cooling": (0.15, 140, "냉각 팬 모터", 3.0)}
KIT_COST = 1.0                                  # 모든 정비에 함께 소모되는 씰·필터 키트 단가지수
BANDS = {"b50": (40, 60), "b100": (90, 110), "b150": (140, 156), "b165": (156, 176), "b200": (190, 215)}


def make_runs(n=48, seed=1):
    """반환: units(개체 정보), snap(일별 측정 요약), W(진동 파형 [측정 수, 256])"""
    rng = np.random.default_rng(seed)
    names = list(MODES)
    mode = rng.choice(names, n, p=[MODES[m][0] for m in names])
    life = np.clip(np.array([rng.normal(MODES[m][1], 18) for m in mode]), 90, 230).astype(int)
    start = np.sort(rng.integers(0, 240, n))
    units = pd.DataFrame({"unit": [f"U{i:02d}" for i in range(n)], "mode": mode, "life": life, "start": start, "base_load": rng.uniform(0.4, 0.75, n)})
    rows = []
    for u in units.itertuples():
        d = np.arange(u.life + 1)
        onset = 0.45 * u.life
        deg = np.clip((d - onset) / (u.life - onset), 0, 1) ** 1.5
        load = np.clip(u.base_load + rng.normal(0, 0.06, len(d)), 0.2, 1.0)
        hi = rng.random(len(d)) < 0.07                                            # 고부하 운용일(정상이지만 진동·온도 상승)
        load = np.where(hi, rng.uniform(0.95, 1.1, len(d)), load)
        rows.append(pd.DataFrame({"unit": u.unit, "day": d, "cal": u.start + d, "rul": u.life - d, "deg": deg, "load": load, "hi_load": hi, "mode": u.mode}))
    snap = pd.concat(rows, ignore_index=True)
    S = len(snap)
    snap["ambient"] = 15 + 12 * np.sin(2 * np.pi * snap["cal"] / 365) + rng.normal(0, 2, S)
    dg, ld, md = snap["deg"].values, snap["load"].values, snap["mode"].values
    isb, isg, iso, isc = (md == "bearing"), (md == "gear"), (md == "oil_pump"), (md == "cooling")
    snap["temp"] = 55 + 22 * ld + 0.5 * snap["ambient"] + 9 * dg * iso + 26 * dg * isc + 3 * dg * (isb | isg) + rng.normal(0, 1.5, S)
    snap["press"] = 450 - 12 * ld - 85 * dg * iso - 15 * dg * isc + rng.normal(0, 6, S)
    ph = rng.uniform(0, 2 * np.pi, (S, 3))
    a1, a2 = 1.0 * (0.5 + ld), 0.4 * (0.5 + ld)
    W = (a1[:, None] * np.sin(2 * np.pi * 50 * TT + ph[:, :1]) + a2[:, None] * np.sin(2 * np.pi * 100 * TT + ph[:, 1:2])
         + rng.normal(0, 0.35, (S, NS)))
    W += (0.9 * dg * isb)[:, None] * np.sin(2 * np.pi * 165 * TT + ph[:, 2:3])
    W += (0.8 * dg * isg)[:, None] * (np.sin(2 * np.pi * 150 * TT + ph[:, 2:3]) + 0.8 * np.sin(2 * np.pi * 100 * TT))
    W += (0.5 * dg * iso)[:, None] * np.sin(2 * np.pi * 200 * TT + ph[:, 2:3])
    k = rng.poisson(10 * dg * isb)                                                 # 베어링 열화: 충격성 펄스(첨도 상승)
    pos, amp = rng.integers(0, NS, (S, 30)), rng.normal(0, 3.0, (S, 30)) * dg[:, None]
    amp = amp * (np.arange(30)[None, :] < k[:, None])
    np.add.at(W, (np.repeat(np.arange(S), 30), pos.ravel()), amp.ravel())
    return units, snap, W


def extract(W):
    """진동 파형 -> 특징량: RMS, 첨도, 파고율, 최대-최소, FFT 대역 에너지(50/100/150/165/200Hz 부근)"""
    x = W - W.mean(axis=1, keepdims=True)
    rms = np.sqrt((x ** 2).mean(axis=1))
    out = {"rms": rms, "kurt": kurtosis(x, axis=1, fisher=False), "crest": np.abs(x).max(axis=1) / rms, "p2p": x.max(axis=1) - x.min(axis=1)}
    A = np.abs(np.fft.rfft(x * np.hanning(NS), axis=1)) / NS
    f = np.fft.rfftfreq(NS, 1 / FS)
    for nm, (lo, hi) in BANDS.items():
        out[nm] = np.sqrt((A[:, (f >= lo) & (f < hi)] ** 2).sum(axis=1))
    return pd.DataFrame(out)

# ---------------------------------------------------------------------------
# 공통 분석 조각: 일별 특징량 표(파형 특징 + 온도·압력 + 파생), 개체 단위 학습/검증 분할(앞선 24대 학습, 뒤 24대 검증)
# ---------------------------------------------------------------------------
BASE = ["rms", "kurt", "crest", "p2p", "b50", "b100", "b150", "b165", "b200", "temp", "press"]
ROLL = ["rms", "kurt", "b150", "b165", "b200", "temp", "press"]


def build_features(snap, W):
    D = pd.concat([snap.reset_index(drop=True), extract(W)], axis=1)
    D["temp_res"] = D["temp"] - (55 + 22 * D["load"] + 0.5 * D["ambient"])        # 부하·외기 대비 온도 잔차
    D["press_res"] = D["press"] - (450 - 12 * D["load"])                           # 부하 대비 압력 잔차
    D["rms_n"] = D["rms"] / (0.5 + D["load"])                                      # 부하 정규화 RMS
    D["hf_ratio"] = (D["b150"] + D["b165"] + D["b200"]) / (D["b50"] + 1e-6)         # 고주파/기본주파 비
    g = D.groupby("unit")
    for c in ROLL:
        D[c + "_m3"] = g[c].transform(lambda s: s.rolling(3, min_periods=1).mean())
    for c in ("rms_n", "temp_res", "press_res"):
        D[c + "_d"] = g[c].diff().fillna(0)
    return D


DERIVED = ["temp_res", "press_res", "rms_n", "hf_ratio"] + [c + "_m3" for c in ROLL] + ["rms_n_d", "temp_res_d", "press_res_d"]
FEATS = BASE + DERIVED
AE_COLS = ["rms_n", "kurt", "b150", "b165", "b200", "hf_ratio", "temp_res", "press_res"]


def split_units(units, n_train=24):
    return list(units["unit"][:n_train]), list(units["unit"][n_train:])


class AE:
    """오토인코더(입력 8 -> 10 -> 4 -> 10 -> 8): 정상 자료만 재구성하도록 학습하고, 재구성 오차를 이상 점수로 쓴다."""

    def __init__(self, cols=AE_COLS, q=0.99, seed=0):
        self.cols, self.q, self.seed = cols, q, seed

    def fit(self, healthy):
        self.sc = StandardScaler().fit(healthy[self.cols])
        Z = self.sc.transform(healthy[self.cols])
        self.net = MLPRegressor(hidden_layer_sizes=(10, 4, 10), max_iter=800, random_state=self.seed).fit(Z, Z)
        self.thr = float(np.quantile(self.err(healthy), self.q))
        return self

    def contrib(self, d):
        Z = self.sc.transform(d[self.cols])
        return (Z - self.net.predict(Z)) ** 2                 # 입력 항목별 재구성 오차(이상 기여도)

    def err(self, d):
        return self.contrib(d).mean(axis=1)


def latch(d, e, thr, k=2):
    """같은 개체에서 이상 점수가 k일 연속 기준을 넘으면 경보를 켜고 유지(래치)한다."""
    out, e = np.zeros(len(d), bool), np.asarray(e)
    for idx in d.reset_index(drop=True).groupby("unit").indices.values():
        run, on = 0, False
        for i in idx:
            run = run + 1 if e[i] > thr else 0
            on = on or run >= k
            out[i] = on
    return out

FA_COST, FA_HOURS, LATE_COST = 1.0, 3.0, 0.15      # [교육용가상자료] 오경보 1회 점검 비용지수, 점검 소요시간(h), 확보 시간 1일 손실 비용지수
units, snap, W = make_runs(seed=5)
D = build_features(snap, W)
order = list(units["unit"])
tr_u, aar_u, val_u = order[:24], order[24:36], order[36:]
tr = D[D["unit"].isin(tr_u)]
ae = AE().fit(tr[tr["deg"] == 0])
healthy_err = ae.err(tr[tr["deg"] == 0])
E = {u: (D[D["unit"] == u].reset_index(drop=True), ae.err(D[D["unit"] == u])) for u in order}


def evaluate(us, q, k, relax):
    """반환: 오경보 에피소드 수, 원인 분류 건수, 고장 전 확보 시간(일) 목록, 미탐 개체 수, 손실 확보 일수"""
    base_thr = float(np.quantile(healthy_err, q))
    fa, causes, leads, miss, lost = 0, {"고부하 운용일": 0, "외기 급변·고온": 0, "센서 스파이크": 0, "열화 초기 경계": 0, "기타": 0}, [], 0, 0.0
    for u in us:
        g, e = E[u]
        thr = base_thr * np.where(g["load"].values > 0.95, 1 + relax, 1.0)
        over = e > thr
        run = np.zeros(len(g), int)
        for i in range(len(g)):
            run[i] = run[i - 1] + 1 if over[i] and i else int(over[i])
        al = run >= k
        an = (g["deg"] >= 0.1).values
        start = al & ~np.r_[False, al[:-1]]
        for i in np.where(start & ~an)[0]:
            fa += 1
            r = g.iloc[i]
            c = "고부하 운용일" if r["hi_load"] else ("외기 급변·고온" if r["ambient"] > 24 else ("센서 스파이크" if r["kurt"] > 3.2 else ("열화 초기 경계" if r["deg"] > 0 else "기타")))
            causes[c] += 1
        first_an = np.argmax(an)
        hit = np.where(al & an)[0]
        ideal = g["rul"].iloc[first_an]
        if len(hit):
            leads.append(int(g["rul"].iloc[hit[0]])); lost += ideal - g["rul"].iloc[hit[0]]
        else:
            miss += 1; lost += ideal
    return fa, causes, leads, miss, lost


def J(res):
    return FA_COST * res[0] + LATE_COST * res[4]


cur = dict(q=0.99, k=1, relax=0.0)               # 현행(AAR 이전) 파라미터
r_aar = evaluate(aar_u, **cur)
print(f"[교육용가상자료] AAR 기간 {len(aar_u)}대: 현행 파라미터(정상 {cur['q']:.0%} 분위, {cur['k']}일, 고부하 완화 {cur['relax']:.0%}) 오경보 {r_aar[0]}건")
tot = sum(r_aar[1].values())
for c, v in sorted(r_aar[1].items(), key=lambda x: -x[1]):
    print(f"  {c:10s} {v:3d}건 ({v / max(tot, 1):.0%})")

# 재보정: AAR 기간 자료만으로 격자 탐색 (미탐 개체가 없는 조합 중 비용 최소)
grid = []
for q, k, rl in itertools.product((0.98, 0.99, 0.995, 0.999), (1, 2, 3), (0.0, 0.3, 0.6)):
    r = evaluate(aar_u, q, k, rl)
    if r[3] == 0:
        grid.append((J(r), q, k, rl))
grid.sort()
best = dict(q=grid[0][1], k=grid[0][2], relax=grid[0][3])
print(f"\n재보정 결과(격자 {4 * 3 * 3}개 중 미탐 0개 조합 {len(grid)}개 탐색): 정상 {best['q']:.1%} 분위, 연속 {best['k']}일, 고부하일 임계 {best['relax']:.0%} 완화")

rows = []
for nm, us in (("AAR 기간(튜닝)", aar_u), ("검증 기간(미사용)", val_u)):
    a, b = evaluate(us, **cur), evaluate(us, **best)
    rows.append((nm, a, b))
    print(f" {nm:14s} 오경보 {a[0]:3d} -> {b[0]:3d}건, 고장 전 확보 중앙값 {np.median(a[2]):.0f} -> {np.median(b[2]):.0f}일, 미탐 {a[3]} -> {b[3]}대")

# 3대 정량 효과 지표(검증 3개월 대신 검증 12대 기준, 연환산은 개체 수 비례)
a, b = rows[1][1], rows[1][2]
jc, jn = J(a), J(b)
print("\n[3대 정량 효과 지표] (검증 12대, 가정 단가)")
print(f" 1) 비용 절감: 비용지수 {jc:.1f} -> {jn:.1f} ({1 - jn / jc:.1%} 절감), 48대 연환산 약 {(jc - jn) * 4:.0f}")
print(f" 2) 시간 단축: 불필요 점검 {a[0] * FA_HOURS:.0f}h -> {b[0] * FA_HOURS:.0f}h ({1 - b[0] / max(a[0], 1):.1%} 단축)")
print(f" 3) 확보 시간: 고장 전 첫 경보 확보 중앙값 {np.median(a[2]):.0f}일 -> {np.median(b[2]):.0f}일, 정상 오경보 {a[0]}건 -> {b[0]}건")
print("[도입 로드맵] 단기(1~3월): 재보정 임계 적용 + 검사관 확인 병행 / 중기(3~12월): 분기 AAR로 임계 재보정, 부하 보정 항 추가 / 장기(1년~): 군수 정보체계 정비 이력과 연동해 임계 자동 갱신")


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] AAR 기간 12대: 현행 파라미터(정상 99% 분위, 1일, 고부하 완화 0%) 오경보 9건
  기타           5건 (56%)
  고부하 운용일      2건 (22%)
  열화 초기 경계     2건 (22%)
  외기 급변·고온     0건 (0%)
  센서 스파이크      0건 (0%)

재보정 결과(격자 36개 중 미탐 0개 조합 36개 탐색): 정상 99.5% 분위, 연속 1일, 고부하일 임계 30% 완화
 AAR 기간(튜닝)     오경보   9 ->   4건, 고장 전 확보 중앙값 58 -> 56일, 미탐 0 -> 0대
 검증 기간(미사용)     오경보  16 ->  10건, 고장 전 확보 중앙값 60 -> 57일, 미탐 0 -> 0대

[3대 정량 효과 지표] (검증 12대, 가정 단가)
 1) 비용 절감: 비용지수 24.7 -> 22.0 (10.9% 절감), 48대 연환산 약 11
 2) 시간 단축: 불필요 점검 48h -> 30h (37.5% 단축)
 3) 확보 시간: 고장 전 첫 경보 확보 중앙값 60일 -> 57일, 정상 오경보 16건 -> 10건
[도입 로드맵] 단기(1~3월): 재보정 임계 적용 + 검사관 확인 병행 / 중기(3~12월): 분기 AAR로 임계 재보정, 부하 보정 항 추가 / 장기(1년~): 군수 정보체계 정비 이력과 연동해 임계 자동 갱신
```
</details>
